# PCAIR multigrid

In the previous two notebooks we established the two ingredients AIR methods need:
1. PMISR-DDC can extract a submatrix $A_{ff}$ that is diagonally dominant.
2. GMRES polynomials can approximate inverses of $A_{ff}$ effectively.

PCAIR combines these ideas into a hierarchy:
- build CF splitting
- approximate ideal transfer operators (especially restriction) using $A_{ff}^{-1}$ approximations
- form a coarse operator
- repeat on the coarse level

The aim is to form a solver/preconditioner that can produce scalable solutions in asymmetric problems, i.e., the amount of work only scales like O(n) with mesh refinement. We would also like to see good weak scaling in parallel, i.e., if the number of unknowns is doubled at the same time as double the compute resources are used on a supercomputer, the time to solve remains the same.  

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt

import petsc4py
petsc4py.init(sys.argv)
from petsc4py import PETSc

import pflare

print("Ready.")

## 1. Why standard methods fail for advection

For upwind advection discretisations, the matrix is typically asymmetric and non-normal, not SPD and not strongly diagonally dominant. This breaks assumptions behind classical smoothers and transfer operators.

Common failure modes:
- Sweep-based methods (Gauss-Seidel, ILU triangular solves) are ordering-sensitive and hard to parallelise, as they must "sweep" across the mesh following the flow.
- Standard AMG components are tuned for SPD-like behavior; convergence can degrade badly on strongly asymmetric operators.
- Block-local preconditioning may lose scalability as decomposition increases.

AIR methods target this regime directly by redesigning both splitting and transfer construction for asymmetric systems, without any of the traditional assumptions which multigrids targetting SPD systems rely on.

## 2. AIR multigrid

If we start with a linear system, we can partition unknowns into F and C points:
$$
\begin{bmatrix} A_{ff} & A_{fc} \\ A_{cf} & A_{cc} \end{bmatrix}
\begin{bmatrix} x_f \\ x_c \end{bmatrix} =
\begin{bmatrix} b_f \\ b_c \end{bmatrix}.
$$

A useful reference point is the exact block LDU factorisation:
$$
A =
\underbrace{\begin{bmatrix} I & 0 \\ A_{cf}A_{ff}^{-1} & I \end{bmatrix}}_{L}
\underbrace{\begin{bmatrix} A_{ff} & 0 \\ 0 & S \end{bmatrix}}_{D}
\underbrace{\begin{bmatrix} I & A_{ff}^{-1}A_{fc} \\ 0 & I \end{bmatrix}}_{U},
\quad S = A_{cc} - A_{cf}A_{ff}^{-1}A_{fc}.
$$

The inverse of this block LDU factorisation is
$$
A^{-1} = U^{-1}D^{-1}L^{-1}
$$
with
$$
U^{-1} = \begin{bmatrix} I & -A_{ff}^{-1}A_{fc} \\ 0 & I \end{bmatrix},
\qquad
D^{-1} = \begin{bmatrix} A_{ff}^{-1} & 0 \\ 0 & S^{-1} \end{bmatrix},
\qquad
L^{-1} = \begin{bmatrix} I & 0 \\ -A_{cf}A_{ff}^{-1} & I \end{bmatrix}.
$$
where we can view components of this inverse as a multi-level method (or equivalently a block Gaussian elimination). 

We take the solution on F points and apply $A_{ff}^{-1}$ as an F-point solve. We then restrict our solution down onto C-points and solve the coarse grid problem $S^{-1}$ formed from a Schur complement. We bring the solution on C-points back up to our top grid and hence we have solved our problem exactly on F points and C points (i.e., our entire grid).  

This is exactly where AIR ideal operators come from:
$$
W = -A_{ff}^{-1}A_{fc}, \qquad Z = -A_{cf}A_{ff}^{-1},
$$
with $P=[W;I]$ and $R=[Z\ I]$, where $P$ is a prolongator and $R$ is a restrictor. 

AIR multigrid is not identical to applying an exact block-LDU inverse, but they share similar features. Rather than form an exact inverse of $A_{ff}$, we approximate it and hence form an approximate ideal restrictor (and/or prolongator).

The key to a scalable AIR method is choosing F points such that we can form cheap but good approximations to $A_{ff}^{-1}$ on each level. Better approximation of $A_{ff}^{-1}$ gives better $W$, $Z$, and F-point smoothing, which improves convergence.

In [ ]:
def build_2d_adv_diff(N, eps=1e-4):
    """2D advection-diffusion: -eps*Lap(u) + u_x = 1 with 5-point FD."""
    h = 1.0 / (N + 1)
    n = N * N
    A = PETSc.Mat().create()
    A.setSizes([n, n])
    A.setFromOptions()
    A.setPreallocationNNZ(5)
    A.setUp()

    rstart, rend = A.getOwnershipRange()
    for row in range(rstart, rend):
        i, j = divmod(row, N)
        diag = 4.0 * eps / h**2 + 1.0 / h
        A.setValue(row, row, diag)
        if j > 0:     A.setValue(row, row - 1, -eps / h**2 - 1.0 / h)
        if j < N - 1: A.setValue(row, row + 1, -eps / h**2)
        if i > 0:     A.setValue(row, row - N, -eps / h**2)
        if i < N - 1: A.setValue(row, row + N, -eps / h**2)
    A.assemblyBegin(); A.assemblyEnd()

    b = A.createVecRight()
    b.set(1.0)
    return A, b

print('Matrix builders ready.')

## 3. AIRG

AIRG specifically uses the GMRES polynomials we discussed previously to form approximations to $A_{ff}^{-1}$. As noted above, the ideal restrictor can be written as $Z = -A_{cf}A_{ff}^{-1}$.

This is the reason for building assembled approximations to the GMRES polynomials; we can then directly use a matrix-matrix product to compute $Z$ easily while retaining the assembled matrix to use as a very strong F-point smoother.  

We can also enable matrix-free option so that it builds the assembled approximation to compute $Z$, then destroys it to save memory. The F-point smooth can then be performed matrix-free by reusing the polynomial coefficients. This can be very efficient on GPU hardware, where repeated matrix-vector products using the same matrix can be fast.

## 4. 2D advection operator

We can now show that PCAIR is very effective on asymmetric systems. By default PCAIR uses:
1. PMISR-DDC to form a C/F splitting
2. AIRG as the AIR method
3. Assembled 6th order GMRES polynomials with 1st order fixed sparsity to approximate $A_{ff}^{-1}$

We run an experiment with a 2D upwinded advection operator and compare the results from AIR with an ILU method and GMRES with no preconditioner. We have deliberately not ordered the unknowns to follow the advection. AIRG and GMRES do not care about unknown ordering, but the ILU method does. If we reorder the unknowns, the ILU method can produce a solution in 1 iteration, but this does not scale in parallel due to the backward/forward substitution.

In [ ]:
def count_iterations(A, b, pc_type, pc_options=None, max_it=500, rtol=1e-10):
    """Run GMRES and return iteration count (unpreconditioned norm).
    Pass PETSc option names exactly as you want them after the `pcair_` KSP prefix,
    e.g. `pc_air_print_stats_timings`, `pc_air_z_type`, `ksp_view`."""
    pc_options = pc_options or {}
    opts = PETSc.Options()
    prefix = 'pcair_'

    set_keys = []
    for key, value in pc_options.items():
        full_key = prefix + key

        is_view_flag = key.endswith('_view')
        if is_view_flag and str(value).strip().lower() in ('1', 'true', 'yes', 'on'):
            opts[full_key] = ''
        elif value is True or value is None:
            opts[full_key] = ''
        else:
            opts[full_key] = str(value)

        set_keys.append(full_key)

    ksp = PETSc.KSP().create()
    ksp.setOptionsPrefix(prefix)
    ksp.setOperators(A)
    ksp.setType('gmres')
    ksp.setNormType(PETSc.KSP.NormType.UNPRECONDITIONED)
    ksp.setTolerances(rtol=rtol, max_it=max_it)
    ksp.getPC().setType(pc_type)
    ksp.setFromOptions()

    x = A.createVecRight()
    x.set(0.0)
    ksp.solve(b, x)

    iters = ksp.getIterationNumber()
    reason = ksp.getConvergedReason()
    if reason < 0:
        iters = max_it

    for key in set_keys:
        try:
            del opts[key]
        except Exception:
            pass

    return iters

# Small practical experiment: strongly asymmetric 1D upwind sequence
grid_sizes = [50, 100, 200, 400]
iters_air = []
iters_ilu = []
iters_none = []

for N in grid_sizes:
    A, b = build_2d_adv_diff(N)

    n_air = count_iterations(
        A, b, 'air',
        max_it=500, rtol=1e-10,
    )

    n_ilu = count_iterations(A, b, 'ilu', max_it=500, rtol=1e-10)
    n_none = count_iterations(A, b, 'none', max_it=1000, rtol=1e-10)

    iters_air.append(n_air)
    iters_ilu.append(n_ilu)
    iters_none.append(n_none)

    A.destroy(); b.destroy()
    print(f'N={N:4d}: PCAIR={n_air:4d}  ILU={n_ilu:4d}  None={n_none:4d}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.loglog(grid_sizes, iters_none, 'k--o', label='No preconditioner')
ax.loglog(grid_sizes, iters_ilu, 'g-^', label='ILU(0)')
ax.loglog(grid_sizes, iters_air, 'r-s', label='PCAIR')
ax.set_xlabel('Problem size N (1D upwind, N unknowns)')
ax.set_ylabel('GMRES iterations to rtol=1e-10 (unpreconditioned norm)')
ax.set_title('Asymmetric advection: iteration growth comparison')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

We can see AIRG performs very well on this problem, with almost flat iteration count. We have not altered the default parameters here at all, often this is required to get the best performance, particularly at scale.

## 4. Hierarchy and varying parameters

We can give the option `-pc_air_print_stats_timings` to view the construction of the hierarchy, output of cumulative timings results and statistics on the resulting multigrid. 

In [ ]:
# Run one PCAIR solve with hierarchy/stat printing enabled
A, b = build_2d_adv_diff(100)

pcair_stats_options = {
    'pc_air_print_stats_timings': '1',
}

n_air_stats = count_iterations(
    A, b, 'air',
    pc_options=pcair_stats_options,
    max_it=500,
    rtol=1e-10,
 )

print(f"PCAIR (with hierarchy stats) iterations = {n_air_stats}")

A.destroy()
b.destroy()

We can see there were 12 levels formed in the multigrid, with some final statistics printed out about the hierarchy at the bottom. In particular, the cycle complexity and the storage complexity are important to note. 

The cycle complexity gives the equivalent number of matrix-vector products of the input matrix required to do one iteration of AIR. Multiplying this by the iteration count gives an indication of the amount of work required to do a solve. In parallel and on GPUs this isn't always an accurate estimate of solve time, but it can be helpful to get a feel for the cost of a solve. 

The storage complexity gives the equivalent number of copies of the input matrix required to store the AIR multigrid. 

Options such as drop tolerances, coarsening rates, matrix-free smoothing, etc, can all affect these statistics. For example, if we turn on matrix-free smoothing, we can see both the cycle complexity go up (as we require more matrix-vector products to apply our GMRES polynomials in the F-point smoothing), but the memory use go down. 

In [ ]:
# Run one PCAIR solve with hierarchy/stat printing enabled
A, b = build_2d_adv_diff(100)

pcair_stats_options = {
    'pc_air_print_stats_timings': '1',
    'pc_air_matrix_free_polys': '1',
}

n_air_stats = count_iterations(
    A, b, 'air',
    pc_options=pcair_stats_options,
    max_it=500,
    rtol=1e-10,
 )

print(f"PCAIR (with hierarchy stats) iterations = {n_air_stats}")

A.destroy()
b.destroy()

## 5. AIR variants

Common AIR variants are available in PCAIR and differ mainly in transfer approximation and smoother/inverse choices:
- AIRG uses GMRES polynomials to approximate $A_{ff}^{-1}$
- nAIR uses Neumann polynomials to approximate $A_{ff}^{-1}$
- lAIR approximates $Z$ directly

For example we can turn on lAIR operators with GMRES polynomial smoothing by specifying `-pc_air_z_type lair`. We can also view the options used in the solve by specifying `ksp_view`. 

In [ ]:
# Run one PCAIR solve with hierarchy/stat printing enabled
A, b = build_2d_adv_diff(100)

pcair_stats_options = {
    'pc_air_print_stats_timings': '1',
    'pc_air_z_type': 'lair',
    'ksp_view': '',
}

n_air_stats = count_iterations(
    A, b, 'air',
    pc_options=pcair_stats_options,
    max_it=500,
    rtol=1e-10,
 )

print(f"PCAIR (with hierarchy stats) iterations = {n_air_stats}")

A.destroy()
b.destroy()

## Summary

- Standard iterative methods can fail on strongly asymmetric, non-normal systems.
- AIR methods reformulate multigrid around F/C reduction and approximate ideal operators.
- AIRG can give scalable solutions in difficult advection problems.